# Pasar Rau Sales Analysis - Data Cleaning

## Objective

Transform the raw sales data into a consistent analytical dataset while
preserving the original business information and documenting all assumptions.

## Cleaning Principles

1. Preserve the raw source unchanged.
2. Convert fields into appropriate analytical data types.
3. Do not fabricate missing or corrupted identifiers.
4. Preserve returns and reversals as valid business activity.
5. Flag questionable records rather than silently deleting them.
6. Remove records only when there is sufficient evidence that they are invalid.
7. Validate important transformations before exporting the processed dataset.

#### Loading The File

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np


RAW_DATA_PATH = Path("../data/raw/Penjualan2021-Pasarawu.csv")


with RAW_DATA_PATH.open("r", encoding="utf-8-sig") as file:
    header_line = file.readline().strip()

columns = header_line.split(",")


df_raw = pd.read_csv(
    RAW_DATA_PATH,
    sep=";",
    skiprows=1,
    names=columns,
    usecols=range(len(columns)),
    dtype=str,
    engine="python"
)


df = df_raw[df_raw["Week"].notna()].copy()


df["Net"] = df["Net"].str.replace(
    r",+$",
    "",
    regex=True
)


assert df.shape == (10678, 29)

df.head()

,Week,Month,Quart,DistID,DistName,Salesman,SalesmaneNama,INVNumber,INVDate,Outlet,...,PriceCase,GSV,Discount,Tax,PERIOD,NOTES 1,NOTES 2,1000000,PJP,Net
0,1,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021000360,06/01/21,367304001000328956,...,105382,210764,-9272,"20149,2",0,RINSO MOLTO40,SKU TARGET,"0,211",Wednesday,"221641,2"
1,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,46364,46364,"-2372,43","4399,16",0,RINSO240,SKU TARGET,"0,046",Wednesday,"48390,73"
2,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,104727,104727,"-5982,53","9874,44",0,WIPOL SST,SKU TARGET,"0,105",Wednesday,"108618,91"
3,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,"98181,99999","36818,25","-3491,84","3332,64",0,RINSO ROYAL20ML,SKU TARGET,"0,037",Wednesday,"36659,5"
4,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,91636,"11454,5","-85,91","1136,86",0,MOLTOBLU280,SKU TARGET,"0,011",Wednesday,"12505,45"


### We Will Rename The Columns So It's Easier To Understand

In [7]:
column_mapping = {
    "Week": "week",
    "Month": "month",
    "Quart": "quarter",
    "DistID": "distributor_id",
    "DistName": "distributor_name",
    "Salesman": "salesman_code",
    "SalesmaneNama": "salesman_name",
    "INVNumber": "invoice_number",
    "INVDate": "invoice_date",
    "Outlet": "outlet_id",
    "Outlet Name": "outlet_name",
    "OutletAddress": "outlet_address",
    "SKUCode": "sku_code",
    "Product Name": "product_name",
    "Case": "case_qty",
    "Dozen": "dozen_qty",
    "Pieces": "piece_qty",
    "Weight": "weight",
    "TotalQuantity(PCS)": "total_quantity_pcs",
    "PriceCase": "price_case",
    "GSV": "gsv",
    "Discount": "discount",
    "Tax": "tax",
    "Net": "net_sales",
    "PERIOD": "period",
    "NOTES 1": "notes_1",
    "NOTES 2": "notes_2",
    "1000000": "gsv_millions",
    "PJP": "pjp",
}

df = df.rename(columns=column_mapping)

In [8]:
df.head()

,week,month,quarter,distributor_id,distributor_name,salesman_code,salesman_name,invoice_number,invoice_date,outlet_id,...,price_case,gsv,discount,tax,period,notes_1,notes_2,gsv_millions,pjp,net_sales
0,1,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021000360,06/01/21,367304001000328956,...,105382,210764,-9272,"20149,2",0,RINSO MOLTO40,SKU TARGET,"0,211",Wednesday,"221641,2"
1,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,46364,46364,"-2372,43","4399,16",0,RINSO240,SKU TARGET,"0,046",Wednesday,"48390,73"
2,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,104727,104727,"-5982,53","9874,44",0,WIPOL SST,SKU TARGET,"0,105",Wednesday,"108618,91"
3,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,"98181,99999","36818,25","-3491,84","3332,64",0,RINSO ROYAL20ML,SKU TARGET,"0,037",Wednesday,"36659,5"
4,2,1,1,93047001,PT. FAJAR PANGAN LESTARI,21,SAEFUL EFENDI,210021001430,13/01/2021,367304001000328956,...,91636,"11454,5","-85,91","1136,86",0,MOLTOBLU280,SKU TARGET,"0,011",Wednesday,"12505,45"


### Convert Dates

In [9]:
df["invoice_date"] = pd.to_datetime(
    df["invoice_date"],
    dayfirst=True,
    errors="coerce"
)

C:\Users\Thomas\AppData\Local\Temp\ipykernel_10024\1231601551.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["invoice_date"] = pd.to_datetime(


In [10]:
df["invoice_date"].isna().sum()

np.int64(0)

In [11]:
df["invoice_date"].min(), df["invoice_date"].max()

(Timestamp('2021-01-04 00:00:00'), Timestamp('2021-12-31 00:00:00'))

### Convert Numeric Fields

In [12]:
numeric_columns = [
    "case_qty",
    "dozen_qty",
    "piece_qty",
    "weight",
    "total_quantity_pcs",
    "price_case",
    "gsv",
    "discount",
    "tax",
    "net_sales",
    "gsv_millions"
]

In [13]:
# We will reuse the parser from notebook 01
def parse_indonesian_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value == "":
        return np.nan

    # If both separators exist, assume:
    # dot = thousands separator
    # comma = decimal separator
    if "." in value and "," in value:
        value = value.replace(".", "").replace(",", ".")

    # If only comma exists, treat comma as decimal separator
    elif "," in value:
        value = value.replace(",", ".")

    return pd.to_numeric(value, errors="coerce")

In [14]:
for col in numeric_columns:
    df[col] = df[col].map(parse_indonesian_number)

In [15]:
df[numeric_columns].dtypes

case_qty                int64
dozen_qty               int64
piece_qty               int64
weight                float64
total_quantity_pcs    float64
price_case            float64
gsv                   float64
discount              float64
tax                   float64
net_sales             float64
gsv_millions          float64
dtype: object

In [16]:
df[numeric_columns].isna().sum()

case_qty              0
dozen_qty             0
piece_qty             0
weight                0
total_quantity_pcs    0
price_case            0
gsv                   0
discount              0
tax                   0
net_sales             0
gsv_millions          0
dtype: int64

### Preserve identifiers as strings
Some columns should never become floats because it's an identifier label and not mathematic values.

In [17]:
identifier_columns = [
    "distributor_id",
    "salesman_code",
    "invoice_number",
    "outlet_id",
    "sku_code",
]

for col in identifier_columns:
    df[col] = df[col].astype("string").str.strip()

### Create an invoice reliability flag

We aren't going to fix some values like `2,1003E+11` yet because we still don't know the missing digits.

Instead, we're going to separate reliable data from the overall data. So downstream analysis can deliberately choose the only dataframe with reliable invoice id for metrics requiring trustworthy invoice numbers.

That's much better than either use corrupted IDs anyway or delete half the dataset.

In [18]:
df["invoice_id_reliable"] = ~df["invoice_number"].str.contains(
    r"[Ee]\+",
    regex=True,
    na=False
)

In [19]:
df["invoice_id_reliable"].value_counts()

invoice_id_reliable
False    7033
True     3645
Name: count, dtype: Int64

### Flag duplicates

In [ ]:
df["is_exact_duplicate"] = df.duplicated(
    subset=[col for col in df.columns if col != "is_exact_duplicate"],
    keep=False
)

In [21]:
duplicate_mask = df.duplicated(keep=False)

df["is_exact_duplicate"] = duplicate_mask

In [22]:
df["is_exact_duplicate"].value_counts()

is_exact_duplicate
False    10636
True        42
Name: count, dtype: int64

### Classify transaction direction

We're going to classify the transactions: return for transactions with negative net transaction value and sale for transactions with positive net transaction value.

In [23]:
def classify_transaction(row):
    if row["total_quantity_pcs"] < 0 and row["net_sales"] < 0:
        return "return"
    elif row["total_quantity_pcs"] > 0 and row["net_sales"] > 0:
        return "sale"
    else:
        return "adjustment_or_mixed"

df["transaction_type"] = df.apply(classify_transaction, axis=1)

In [24]:
df["transaction_type"].value_counts()

transaction_type
sale                   9813
return                  702
adjustment_or_mixed     163
Name: count, dtype: int64

### Financial fields

In [25]:
df["sales_after_discount"] = (
    df["gsv"] + df["discount"]
)

In [26]:
df["discount_rate"] = np.where(
    df["gsv"].abs() > 0,
    df["discount"].abs() / df["gsv"].abs(),
    np.nan
)

### Don't normalize quantity units yet

We will be leaving these alone: 
`Case` `Dozen` `Pieces` `TotalQuantity(PCS)` `PriceCase`

Because we already have: `total_quantity_pcs` which appears intended as the normalized piece quantity.

Until we understand package-size semantics, we don't fabricate conversion logic.

### Validation checks

In [27]:
assert len(df) == 10678

In [28]:
print("Rows:", len(df))
print("Date nulls:", df["invoice_date"].isna().sum())
print("SKU nulls:", df["sku_code"].isna().sum())
print("Net nulls:", df["net_sales"].isna().sum())
print("Exact duplicate rows:", df["is_exact_duplicate"].sum())

Rows: 10678
Date nulls: 0
SKU nulls: 0
Net nulls: 0
Exact duplicate rows: 42


Financial check. Expected output is < 1.

In [29]:
net_diff = (
    df["net_sales"]
    - (df["gsv"] + df["discount"] + df["tax"])
)

net_diff.abs().max()

np.float64(0.8100000005215406)

### Export the cleaned dataset

In [30]:
OUTPUT_PATH = Path("../data/processed/pasar_rau_sales_clean.csv")

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

OUTPUT_PATH

WindowsPath('../data/processed/pasar_rau_sales_clean.csv')